In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/DTD-Wijesinghe/IT3091---Machine-Learning-Assignment.git"
STUDENT_ROLE = "IT23361690 - Gunathilake B.M.P"
STUDENT_NAME = input("Enter your real GitHub display name: ").strip()
STUDENT_EMAIL = input("Enter your verified GitHub email: ").strip()
CLONE_DIR = Path("/content/IT3091---Machine-Learning-Assignment")

if not CLONE_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(CLONE_DIR)], check=True)
os.chdir(CLONE_DIR)
subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "checkout", "main"], check=True)
subprocess.run(["git", "pull", "--ff-only", "origin", "main"], check=True)
subprocess.run(["git", "config", "user.name", STUDENT_NAME], check=True)
subprocess.run(["git", "config", "user.email", STUDENT_EMAIL], check=True)
print("Ready for", STUDENT_ROLE)
print("Authenticate your own GitHub account before running the final cell.")

# Model training and project outputs

This notebook runs after the first three contributions. It trains the yield and Colombo price models from the generated preprocessed datasets and saves the artefacts and metrics.

In [ ]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd()
models_dir = ROOT / 'models'
evidence_dir = ROOT / 'evidence'
models_dir.mkdir(exist_ok=True)
evidence_dir.mkdir(exist_ok=True)

def make_pipeline(categorical, numeric):
    prep = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical), ('num', StandardScaler(), numeric)])
    return Pipeline([('prep', prep), ('model', RandomForestRegressor(n_estimators=150, min_samples_leaf=2, random_state=42, n_jobs=-1))])

def train(df, target, date_col, categorical, numeric, task, filename):
    df = df.sort_values(date_col).copy()
    years = pd.to_datetime(df[date_col]).dt.year
    train_df = df[years <= 2019]
    test_df = df[years >= 2020]
    if len(train_df) == 0 or len(test_df) == 0:
        cut = int(len(df) * 0.8)
        train_df, test_df = df.iloc[:cut], df.iloc[cut:]
    columns = categorical + numeric
    model = make_pipeline(categorical, numeric)
    model.fit(train_df[columns], train_df[target])
    prediction = model.predict(test_df[columns])
    metrics = {'Task': task, 'Records': len(df), 'TrainRecords': len(train_df), 'TestRecords': len(test_df), 'MAE': mean_absolute_error(test_df[target], prediction), 'RMSE': mean_squared_error(test_df[target], prediction) ** 0.5, 'R2': r2_score(test_df[target], prediction)}
    joblib.dump(model, models_dir / filename)
    return metrics

yield_df = pd.read_csv(ROOT / 'data' / 'preprocessed' / 'crop_yield_preprocessed.csv')
price_df = pd.read_csv(ROOT / 'data' / 'preprocessed' / 'colombo_weekly_prices_preprocessed.csv', parse_dates=['Date', 'NextWeekStart'])
yield_metrics = train(yield_df, 'Yield_t_ha', 'ForecastDate', ['District', 'CropCategory', 'Crop', 'Season'], ['Year', 'Rainfall_3M_mm', 'TempMean_3M_C', 'WindSpeed_3M_kmh', 'SolarRadiation_3M_MJ_m2', 'ET0_3M_mm', 'WeatherDays_3M'], 'Yield', 'crop_yield_model.joblib')
price_metrics = train(price_df, 'NextWeekWholesalePrice_RsKg', 'Date', ['Crop', 'Market'], ['Year', 'Month', 'WeekOfYear', 'CurrentPrice_RsKg', 'Lag1Price_RsKg', 'Lag2Price_RsKg', 'Lag3Price_RsKg', 'Rolling7Mean_RsKg', 'Rolling28Mean_RsKg', 'Rolling7Std_RsKg'], 'Colombo price', 'wholesale_price_model.joblib')
metrics = pd.DataFrame([yield_metrics, price_metrics])
metrics.to_csv(evidence_dir / 'model_metrics.csv', index=False)
print(metrics.to_string(index=False))
print('Saved model files and evidence/model_metrics.csv')

The saved models are trained from the generated weekly-price and weather-aligned yield tables. The final application can load these files after the last push.

In [ ]:
import subprocess

print(subprocess.run(["git", "status", "--short"], capture_output=True, text=True).stdout)
subprocess.run(["git", "add", "notebooks/04_model_training_gunathilake.ipynb"], check=True)
subprocess.run(["git", "add", "models/crop_yield_model.joblib"], check=True)
subprocess.run(["git", "add", "models/wholesale_price_model.joblib"], check=True)
subprocess.run(["git", "add", "evidence/model_metrics.csv"], check=True)
subprocess.run(["git", "commit", "-m", "Train models from prepared datasets"], check=False)
subprocess.run(["git", "push", "origin", "main"], check=True)
print("Contribution pushed to main.")